# Unit I: Climate Change Module

## Reproducing the hockey stick, and checking whether we got it right

Our target is the result reported by [Mann et al (1998)](https://doi.org/10.1038/33859): that
recent warming stands outside the range of natural variability over the past several centuries.
You will rebuild that picture from current observations and ice-core records, in the style of
NASA's [vital signs](https://climate.nasa.gov/vital-signs).

A language model will write most of the parsing code. It is good at it, and it is also wrong in
ways that run cleanly and produce a plot. Your job is the question scientists have always had to
answer about code they did not write:

> **How do I know these numbers are right?**

Each part below gives you a data source and a question. It does not tell you what is wrong with
the data. Finding that out is the assignment.

---

# Part 1: The code you didn't write

Mauna Loa monthly CO2: <https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt>

Below is working code for that file. It runs and it produces a plausible plot.

In [ ]:
import pandas as pd
from plotnine import *

: 

In [ ]:
columns = ['year', 'month', 'decimal_date', 'average', 'deseasonalized',
           'ndays', 'stdev', 'unc']
df = pd.read_csv("https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt",
                  sep = r'\s+',
                  comment = '#',
                  header = None,
                  names = columns
                  )
df


In [ ]:
ggplot(df, aes(x="decimal_date", y="average")) + geom_line()

### Task 1.1: Read the source, not the code

That `columns` list was written by hand. Open the raw file and read its header block, then check
every name against what the header says the column holds. Report any that misstate it, and say
which of those would change a published figure if nobody noticed.

**Your answer:**
The raw file only has 7 explicit columns, before my AI assitant gave me an answer saying it had 8 columns in its commetn block. The mislabeled columns are column 5, 6, 7, and 8. This impacts the provided plot figure for accurate results. Specifically, column 5 contains monthly inaccurate values, causing the figure to reflect the jagged picks instead of a smooth trendline. 


### Task 1.2: Missing values

How does this file encode a measurement that was not made? Verify it in the raw file, count the
affected rows per column, and say what a naive `mean()`, `min()`, or error bar would report if you
left them in. State what you did about it and why.

In [ ]:
df[l"std_days", "uncertainty", "empty"|] = df[I"std_days", "uncertainty", "empty"|].where(dfIl"std_days", "uncertainty", "empty"|] >= )

: 

In [ ]:
df = df.dropna()

: 

**Your answer:**
According to the # comment block in the raw NOAA co2_mm_mlo.txt header file, unrecorded or missing monthly average CO2 measurements are assigned a placeholder sentinel value of -99.99 in the average column. Additionally, months where individual daily measurement counts were not recorded (specifically prior to May 1974) encode the missing day counts using a placeholder value of -1 in the #days column. In the raw dataset, there are 7 rows with -99.99 in the average column and 194 rows with -1 in the #days column. The Naive min() -99.99 as the lowest value instead of its true minimum (313-314 ppm). Naive mean() the overall average would be around 361 ppm to 354 ppm. The error bars/ plotting contained -99.99 which cause the plots y-axis to stretch far be;low zero. To fix this issue, all instances of -99.99 and -1 were converted to NaN using pandas. 



### Task 1.3: What the numeric columns cannot tell you

The header block contains prose notes as well as column definitions. Read them.

Does anything there change how you would plot or interpret this series? Show your evidence in the
data rather than asserting it, and say whether a model reading only the numeric columns would have
any way to know.

The prose notes in the header block that `comment='#'` threw away

In [ ]:
import urllib.request

lines = urllib.request.urlopen("https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt").read().decode().splitlines()
prose = [l for l in lines if l.startswith("#")]
"\n".join(prose[24:39])

: 

In [ ]:
import pandas as pd
import numpy as np

url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.txt"
cols = ["year", "month", "decimal_date", "average", "interpolated", "trend", "ndays"]

# Load raw dataset directly using Pandas comment stripping
df_evidence = pd.read_csv(
    url, 
    sep=r'\s+', 
    comment='#', 
    header=None, 
    names=cols
)

# EVIDENCE 1: Missing raw observations ('average' < -90) filled in by 'interpolated'
print("=== EVIDENCE 1: Missing Averages Filled by Interpolation ===")
missing_avg = df_evidence[df_evidence["average"] < -90]
print(missing_avg[["year", "month", "decimal_date", "average", "interpolated", "trend"]].head())

# EVIDENCE 2: Seasonal Cycle Extracted (Interpolated minus Trend)
print("\n=== EVIDENCE 2: Seasonal Cycle Extracted (Interpolated minus Trend) ===")
valid_data = df_evidence[df_evidence["average"] > 0].copy()
valid_data["seasonal_diff"] = (valid_data["interpolated"] - valid_data["trend"]).round(2)
print(valid_data[["year", "month", "average", "interpolated", "trend", "seasonal_diff"]].head(6))

**Your answer:**



### Task 1.4: Check whether a better door exists

Before you fix a parser, find out whether you need one. NOAA distributes this same monthly record
in more than one format: <https://gml.noaa.gov/ccgg/trends/data.html>

Load an alternative distribution. Which of the problems you found above does switching formats
solve, and which does it not?

Load the CSV version — the columns tell you their own names

In [ ]:
co2_csv = pd.read_csv("https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv", comment = '#')
co2_csv.head()

Checking whether switching formats fixed the flags: rows with a negative `sdev`

In [ ]:
co2_csv[co2_csv.sdev < 0]

**Your answer:**

The CSV solved my column-name problem, not the data's problems. It carries a real header row (`year,month,decimal date,average,deseasonalized,ndays,sdev,unc`), so the columns come with their own names - no hand-written `columns` list, nothing mislabeled as "smooth" or "empty", and no bogus 8th blank column like in Tasks 1.1/1.2.

The evidence cell shows what did not change: `sdev < 0` still matches exactly 196 rows, the same as in the `.txt` - all 194 SIO rows from Mar 1958 through Apr 1974 still sit on the negative placeholders (-1, -9.99), and Dec 1975 and Apr 1984 are still flagged only by a negative sdev while their `average` values look normal, so the series still shows no visible gap. The prose notes (SIO credit, the interpolation policy, the eruption site swap) are still `#` comment lines, and `comment='#'` throws them away in this format too.

So switching format fixed my parser, not the data.

### Task 1.5: The other library

Now try to load the original `.txt` with `ibis` instead of `pandas`. Report what happens.

Explain the result in terms of the *file*, not the library. What does it do to a rule like "always
use `ibis`"?

**Your answer:**

### 🔍 Verification block 1

1. Extent: The data represents monthly mean atmospheric carbon dioxide concentrations measured at Mauna Lao Observatory in Hawaii. The data that we are observing is not a direct measurement of global carbon dioxide everywhere, instead the measurements represent background atmospheric conditions.
2. Missing data:Missing values are coded using negative sentinel values instead of NaN. These values need to be replaced by NaN before computation is completed. If they are not replaced, completing functions including mean() and min() would create incorrect average based on the negative value observations currently included. 
3. Units: Atmospheric carbon dioxide is reported in parts per million (ppm) of dry air.
4. Completeness: The combined record begins in 1958, however, it is not a completely uniform observational series.
5. Cross-check: The cleaned dated, with updated values, should show a long term increase in cardon dioxide.

---

# Part 2: Arctic sea ice

National Snow and Ice Data Center, sea ice index G02135:

- Documentation: <https://nsidc.org/data/G02135>
- Data directory: <https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/>

### Task 2.1: Ask first, look second

Before opening that directory yourself, ask your model to load Arctic sea ice extent and plot the
trend. Let it produce a plan.

**Record the prompt you used:**

```
[Load the Arctic sea ice extent and plot the trend, give me an exact plan.]
```

### Task 2.2: What did it actually load?

Now open the data directory and look at what is there.

Check the URL the model's code used against the directory listing *before* you run anything. Did
it point at a file that exists? If it ran, how much of the record does the plot actually show, and
is the title honest? If it failed, would you have predicted that failure from reading the code?

A model that has never seen this directory still has to produce a filename. Note where it got one.

**Your answer:**

The directory at `https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/` contains 12 individual CSV files, one per month: `N_01_extent_v4.0.csv` through `N_12_extent_v4.0.csv`. There is no single combined file. A model prompted to "load Arctic sea ice extent" without seeing the directory would have to guess a filename. A likely guess would be `N_09_extent_v4.0.csv` (September, the month of minimum extent), and this file does exist at the expected URL.

That file, however, contains only September data — 47 rows from 1979 to 2025, one row per year. If a model loaded only this file and titled its plot "Arctic sea ice extent trend," the plot would show 47 points and look plausible, but it would not represent the full annual cycle. The title would be technically honest about September but misleading about what "sea ice extent" conventionally means.

The correct approach is to load all 12 monthly files and combine them. The complete record has 574 rows (1978–2026) covering all 12 months. Columns are `year`, `mo`, `source_dataset`, `region`, `extent`, `area`. The `extent` column is in million km². Some rows use `-9999` as a sentinel for missing data (2 in `extent`, 3 in `area`, 2 in `source_dataset`), which must be replaced with NaN before any computation.

### Task 2.3: Build the complete record

Load the full dataset and plot sea ice extent over time.

Then make a scientific decision and defend it: the September minimum is the standard index for
Arctic sea ice loss, and it is not the same thing as the annual mean. Which does your analysis
report, and why is that right for the question you are asking?
import pandas as pd
import io
import urllib.request
from plotnine import *

In [ ]:

base_url = "https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/"
dfs = []
for m in range(1, 13):
    url = base_url + f"N_{m:02d}_extent_v4.0.csv"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    content = urllib.request.urlopen(req, timeout=30).read().decode("utf-8")
    df = pd.read_csv(io.StringIO(content))
    df.columns = df.columns.str.strip()
    df["month_name"] = df["mo"].map({
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
        7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
    })
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data = data.replace(-9999, float("nan"))
data = data.dropna(subset=["extent"])

data.groupby("mo")["extent"].count()

In [ ]:
sep_data = data[data["mo"] == 9].copy()
sep_year = sep_data.groupby("year")["extent"].first().reset_index()
annual_mean = data.groupby("year")["extent"].mean().reset_index()
annual_mean.columns = ["year", "extent"]

sep_year

**Trends computed from the data:**

- September minimum: approximately -0.76 million km² per decade (~-13% per decade)
- Annual mean: approximately -0.48 million km² per decade (~-4% per decade)

The September minimum trend is roughly three times steeper than the annual mean trend. The September rate is the appropriate metric for Arctic sea ice loss.

In [ ]:
import pandas as pd
import io
import urllib.request
from plotnine import *

In [ ]:
(ggplot(sep_year, aes(x="year", y="extent")) +
    geom_line(color="navy") +
    geom_point(color="navy", size=2) +
    geom_smooth(method="lm", se=True, color="red", linetype="dashed") +
    labs(
        x="Year",
        y="September Sea Ice Extent (million km²)",
        title="September Arctic Sea Ice Minimum Extent (1979–2025)"
    ) +
    theme_minimal())

**Your answer:**

The analysis reports the **September minimum** sea ice extent, not the annual mean. The September minimum is the standard index for Arctic sea ice loss because it captures the peak of the Arctic melt season — the lowest point of the annual cycle — which is the most climatologically meaningful single value for tracking ice loss. The annual mean dilutes the seasonal signal by averaging winter maximums (~14 million km²) with summer minimums (~4 million km²), producing a value (~11 million km²) that never actually occurs in nature. The trend in the annual mean (-0.48 million km² per decade) is real but understates the rate of September decline.

The September minimum shows a decline of approximately -0.76 million km² per decade — a substantial and accelerating loss. The 2012 minimum of 3.57 million km² was a record low, and recent years (2023–2025) remain well below the 1980s levels.

### 🔍 Verification block 2

1. **Extent:** Arctic Ocean sea ice extent, approximately 25°N and northward, in the Northern Hemisphere. The data covers the entire Arctic basin (region = "N").

2. **Missing data:** Missing measurements are encoded as `-9999` in the `extent` and `area` columns (2 rows in extent, 3 in area), and as `-9999` in `source_dataset` (2 rows). These are sentinel values, not real measurements. They were replaced with `NaN` and rows with missing extent were dropped before analysis. A naive `mean()` would include `-9999` values, producing impossible negative averages. A naive `min()` would return `-9999`.

3. **Units:** `extent` is in million km² (million square kilometers), confirmed by the NSIDC documentation and by sanity-checking values (range ~3.5–15 million km², consistent with known Arctic sea ice extent).

4. **Completeness:** The complete record covers 1978–2026 across all 12 months (574 rows total). September data specifically spans 1979–2025 (47 rows). No September rows have missing extent values. Some winter months in the late 1980s have missing data.

5. **Cross-check:** The September minimum trend of approximately -13% per decade (decline of ~0.76 million km² per decade relative to a mean of ~5.9 million km²) is consistent with NSIDC's published estimate of approximately -13% per decade for September sea ice extent (1979–2024). The record 2012 minimum of 3.57 million km² matches the widely reported NSIDC record low. The general declining trend visible in the plot matches published sea ice records from NASA and NSIDC.

---

# Part 6: Reflection

1. **Failure inventory.** List every case where the model produced code that ran without error but
   was wrong. What was wrong, and what tipped you off?
2. **Detection.** Which of those would you have caught from the plot alone? Which required going
   back to the source documentation?
3. **Division of labor.** What did you contribute that the model did not? If the answer for some
   task is "nothing", say so.
4. **Plan mode.** Where did approving the plan catch something? Where was it just friction?
5. **Tool choice.** You have now seen the same task done two ways, and data read locally versus
   streamed. What decides which is right, and who makes that decision?

**Your reflection:**

1. In part 1, I asked for assistance on 1.3, to double check the accuracy of kilo code I used gemini, which gemini told me that my code was completely wrong and that my assistant was hallucinating. But when i ran the gemini's code the evidence to support my answer was missing values. Whereas Kilo Codes code ran perfectly because it included an 8th column and gemini only included 7.
2. Again part 1, I was unsure which ai assistant was giving the proper information which made me go back to the data determine who is right and to best achieve my problem solving.
3. Clarification, and lots of it. the model did give some responses that were very confusing and was to double checking itself with my responses. Also in part 2, I was asking for clear direction on importing pandas and exactly where to place its code.
4. I approved every decision and it seemed the Kilo code kept coding and correcting itself from the previous ask.
5. This is a question I kept going back and forth with while wrapping this module up, which ultimately made me decide that muself the user is right. I can read the raw data myself and create the code that would give me the proper answers.

---

## Submission

- [ ] Notebook runs top to bottom without errors
- [ ] All three verification blocks completed
- [ ] Prompts recorded where asked (Tasks 2.1, 3.1)
- [ ] Memory and timing table filled in (Task 3.3)
- [ ] Stressor selection defended (Task 4.3) and reconciliation quantified (Task 4.4)
- [ ] Hockey stick figure with the modern record in geological context (Task 5.2)
- [ ] Reflection complete (Part 6)
- [ ] README updated with team members, badge, and description
- [ ] Repository passes its GitHub Actions check

See [rubric.md](rubric.md) for scoring.